# Sorting: minimize number of swaps
Assume we have an unsorted string `BDADDDBCCC`. Sort it by doing swaps `x[i] <=> x[j]`. Minimize the number of swaps needed.

Here we use ortools.

We use two different approaches:
* shortest path though a min cost flow network solver
* constraint programming model via the CP-SAT solver

In [1]:
# imports
from sympy.utilities.iterables import multiset_permutations 
from ortools.graph.python import min_cost_flow
from ortools.sat.python import cp_model


In [2]:
# input
start = 'BDADDDBCCC' # initial configuration
final = 'ABBCCCDDDD' # final, sorted configuration

## How many swaps does a simple QuickSort algorithm take?

In [3]:
# super simple QuickSort from: https://www.w3schools.com/dsa/dsa_algo_quicksort.php
# I added some logic to count the number of swaps

def partition(array, low, high):
    swaps = 0
    pivot = array[high]
    i = low - 1

    for j in range(low, high):
        if array[j] <= pivot:
            i += 1
            array[i], array[j] = array[j], array[i]
            swaps += 1

    array[i+1], array[high] = array[high], array[i+1]
    swaps += 1
    return swaps,i+1

def quicksort(array, low=0, high=None): 
    if high is None:
        high = len(array) - 1

    if low < high:
        n1,pivot_index = partition(array, low, high)
        n2 = quicksort(array, low, pivot_index-1)
        n3 = quicksort(array, pivot_index+1, high)
        return n1+n2+n3
    else:
        return 0
    
# sort characters in string 
# we convert list of chars <-> string

def list2str(L):
    return ''.join(L)

def strsort(s):
    L = list(s)
    n = quicksort(L)
    s2 = list2str(L)
    return s2,n

# sort input string
s = start  
t,n=strsort(s)
print(f"{s} -> {t} #swaps:{n}")

# sort again (already sorted)
# this can be bad when using QuickSort
t2,n=strsort(t)
print(f"{t} -> {t2} #swaps:{n}")


BDADDDBCCC -> ABBCCCDDDD #swaps:28
ABBCCCDDDD -> ABBCCCDDDD #swaps:54


## Create (directed) network
The nodes are all possible configurations. The arcs represent how we can go from one configuration to another using a single swap.

In [4]:
# enumerate the nodes

nodes=list(multiset_permutations(start))
# convert lists to strings
nodes = [list2str(lst) for lst in nodes]
nnodes = len(nodes)
print(f"number of nodes: {nnodes=}")

# add a mapping from node id (string) to node number
nodenumber = {nodes[i]:i for i in range(len(nodes))}


number of nodes: nnodes=12600


In [5]:
# enumerate the arcs

arcs={}
narcs = 0
for n in nodes:
    s = set()  # no duplicates
    lst = list(n)
    nlen = len(lst)
    for i in range(nlen-1):
        for j in range(i+1,nlen):
            if lst[i] != lst[j]:
                lst2 = lst.copy()
                lst2[i],lst2[j] = lst[j],lst[i]
                str2 = list2str(lst2)
                s.add(str2)
    narcs += len(s)
    arcs[n]=s.copy()
print(f"number of arcs: {narcs=}")

# organization:
# arcs['BDADDDBCCC'] = {'ADBDDDBCCC', 'BADDDDBCCC', ...}


number of arcs: narcs=441000


## Solve with network solver

In [6]:
smcf = min_cost_flow.SimpleMinCostFlow()

arc_dict = {} # used to interpret solution
for n in nodes:
   n1 = nodenumber[n]
   for nn in arcs[n]:
       n2 = nodenumber[nn] 
       n3 = smcf.add_arc_with_capacity_and_unit_cost(n1,n2,1,1) 
       arc_dict[n3] = (n1,n2)

smcf.set_node_supply(nodenumber[start],1)
smcf.set_node_supply(nodenumber[final],-1) 

status = smcf.solve()
print(f"Status: {status}")
if status == smcf.OPTIMAL:
    print(f"Minimum cost: {smcf.optimal_cost()}")


Status: Status.OPTIMAL
Minimum cost: 5


In [7]:
# form shortest path

next = {arc_dict[i][0]:arc_dict[i][1] for i in range(narcs) if smcf.flow(i) > 0}

n1 = nodenumber[start]
while True:
    n2 = next.get(n1,-1)
    if n2<0: break
    print(f"{nodes[n1]} -> {nodes[n2]}")
    n1 = n2

BDADDDBCCC -> BDADCDBDCC
BDADCDBDCC -> BDACCDBDDC
BDACCDBDDC -> ADBCCDBDDC
ADBCCDBDDC -> ADBCCCBDDD
ADBCCCBDDD -> ABBCCCDDDD


## CP-SAT Model

In [9]:
model = cp_model.CpModel()

maxit = 10
L = len(start)

# --------- variables

# x[k,i] integers for k = 0,1,2,..,maxit
#                     i = 0,1,..., L-1  (here L=len(start))
x = {}
for k in range(maxit+1):
    for i in range(L):
        x[k,i] = model.new_int_var(-1000,+1000,f"x_{k}_{i}")  

# s[k,i] ∈ {0,1}  for k = 0,1,2,..,maxit
#                     i = 0,1,..., L-1  (here L=len(start))
s = {}
for k in range(maxit+1):
    for i in range(L):
        s[k,i] = model.new_bool_var(f"s_{k}_{i}")

# d[k] ∈ {0,1} for k = 0,1,2,..,maxit
d = [model.new_bool_var(f"d_{k}") for k in range(maxit+1)]

# --------- constraints

# x[0,i] = start[i]

for i in range(L):
    model.Add(x[0,i] == ord(start[i]))

# d[k]=0 => sum(i, s[k,i]) = n-2

for k in range(maxit+1):
    model.Add(sum([s[k,i] for i in range(L)]) == L-2).OnlyEnforceIf(d[k].Not())

# d[k]=1 => sum(i, s[k,i]) = n

for k in range(maxit+1):
    model.Add(sum([s[k,i] for i in range(L)]) == L).OnlyEnforceIf(d[k])

# s[k,i]=1 => x[k,i] = x[k-1,i]

for k in range(1,maxit+1):
    for i in range(L):
        model.Add(x[k,i] == x[k-1,i]).OnlyEnforceIf(s[k,i])

# s[k,i]=0 and s[k,j]=0 => x[k,i] = x[k-1,j]

for k in range(1,maxit+1):
    for i in range(L):
        for j in range(L):
            if i != j:
                model.Add(x[k,i] == x[k-1,j]).OnlyEnforceIf(s[k,i].Not(),s[k,j].Not())

# d[k]=1 => x[k-1,i] >= x[k-1,i-1] 

for k in range(1,maxit+1):
    for i in range(1,L):
        model.Add(x[k-1,i] >= x[k-1,i-1]).OnlyEnforceIf(d[k])

model.Minimize(sum([(1-d[k]) for k in range(1,maxit)]))        

# --------- solve

solver = cp_model.CpSolver()
solver.parameters.log_search_progress = True
solver.parameters.num_search_workers = 8

status = solver.solve(model)
print(solver.response_stats())    

for k in range(maxit):
    if (k==0) or (solver.value(d[k])==0):
        s = ''
        for i in range(L):
            s += chr(solver.value(x[k,i])) 
        print(f"{k:2}. {s}")

CpSolverResponse summary:
status: OPTIMAL
objective: 5
best_bound: 5
integers: 219
booleans: 150
conflicts: 6
branches: 340
propagations: 1272
integer_propagations: 1487
restarts: 300
lp_iterations: 0
walltime: 0.259604
usertime: 0.259604
deterministic_time: 0.68202
gap_integral: 0.00523947
solution_fingerprint: 0xe02c76fde00d5fd8

 0. BDADDDBCCC
 1. ADBDDDBCCC
 2. ABBDDDDCCC
 3. ABBDDCDCDC
 4. ABBCDCDCDD
 5. ABBCCCDDDD
